In [1]:
# for making API requests
import requests
# showing Maps
import folium
import folium.plugins
# standard data processing and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(15, 4))

In [2]:
def get_deployments():
    '''Retrieve a table of all sensor deployment locations.

    '''
    # query the API
    deployments = requests.get("https://api.floodnet.nyc/api/rest/deployments/flood").json()
    if 'error' in deployments:
        raise RuntimeError(str(deployments))

    # create a pandas table
    df = pd.DataFrame(deployments['deployments'])

    # drop any sensors without a location (TODO: do this within the query)
    df = df.dropna(subset=['location'])

    # convert date strings to datetime objects
    df['date_deployed'] = pd.to_datetime(df.date_deployed,format = 'ISO8601').dt.tz_localize(tz='America/New_York')
    df['date_down'] = pd.to_datetime(df.date_down,format = 'ISO8601').dt.tz_localize(tz='America/New_York')

    # extract latitude, longitude pair
    df['coordinates'] = [np.array(x['coordinates'][::-1]) for x in df.location]
    return df

In [3]:
# get a dataframe of flood deployments
df_deployments = get_deployments()

# show a summary
print(len(df_deployments), "results")
df_deployments.head().sort_values('date_deployed')

337 results


,name,deployment_id,date_deployed,date_down,deploy_type,location,image,sensor_mount,mounted_over,sensor_status,coordinates
4,BK - Hoyt St/5th St,daily_new_falcon,2020-10-05 00:00:00-04:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",NaN,streetsign post,sidewalk,good,"[40.67667202, -73.99459095]"
1,BK - 9th St/Smith St,overly_heroic_squid,2021-12-14 11:43:00-05:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",47216a15-e74d-4d99-adf5-7b445a57b5b1,streetsign post,sidewalk,good,"[40.673401, -73.994892]"
3,M - E 108th St/1st Ave,easily_fond_kiwi,2024-05-10 00:00:00-04:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",NaN,streetsign post,sidewalk,good,"[40.790943, -73.939451]"
2,BK - E 57th St/Snyder Ave,early_neat_satyr,2024-08-15 12:46:00-04:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",d3637b9a-a529-4f57-8ac0-65efb9dfce17,streetsign-post,sidewalk,good,"[40.65058521103148, -73.92320082577001]"
0,Q - Brookville Blvd/ Snake Rd 1,definitely-neat-hippo,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",d5e56c9e-2e3b-4cdb-9ade-8888ec684e00,utility-pole,dirt,good,"[40.6439, -73.74448]"


In [4]:
df_deployments['date_deployed'] = pd.to_datetime(df_deployments['date_deployed'])

df_deployments['deploy_date'] = df_deployments['date_deployed'].dt.date
df_deployments['deploy_time'] = df_deployments['date_deployed'].dt.time
df_deployments['deploy_tz']   = df_deployments['date_deployed'].dt.tz

In [5]:
df_deployments[['borough', 'intersection']] = df_deployments['name'].str.split(" - ", n=1, expand=True)
df_deployments[['street1', 'street2']] = df_deployments['intersection'].str.split("/", n=1, expand=True)
df_deployments['latitude']  = df_deployments['coordinates'].apply(lambda x: x[0])
df_deployments['longitude'] = df_deployments['coordinates'].apply(lambda x: x[1])

In [6]:
def query_depth_data(deployment_id, start_time, end_time):
    data = requests.get(f"https://api.floodnet.nyc/api/rest/deployments/flood/{deployment_id}/depth", params={
        'start_time': start_time,
        'end_time': end_time,
    }).json()
    if 'error' in data:
        raise RuntimeError(data)
    # lets create a pandas dataframe
    df_depth = pd.DataFrame(data['depth_data'], columns=['deployment_id', 'time', 'depth_proc_mm'])

    # drop missing depth values
    df_depth = df_depth.dropna(subset=['depth_proc_mm'])

    # add the sensor name to the table
    df_depth['name'] = df_depth.deployment_id.apply(lambda name: sensor_name_lookup.get(name))


    # convert the time string to a datetime object
    df_depth['time'] = pd.to_datetime(df_depth['time'], format='ISO8601')

    # convert millimeters to inches
    df_depth['depth_inches'] = df_depth['depth_proc_mm'] / 25.4

    # use time as the dataframe index
    df_depth = df_depth.set_index('time')
    return df_depth

def query_depth_data_for_multiple_deployments(df, start_time, end_time):
    # filter out sensors that were not online during the event
    df_active = df
    #  [
    #     (df.date_deployed < pd.to_datetime(start_time))
    # ]

    # query depth data for all sensors
    df_depth = pd.concat([
        query_depth_data(deployment_id, start_time, end_time)
        for deployment_id in df_active.deployment_id
    ])

    df_final = pd.merge(
        df_depth.reset_index(), # Resets 'time' index to a regular column
        df,                     # Merges with the deployment dataset you passed in
        on='deployment_id', 
        how='left'
    )

    return df_final
def draw_depth_data(df_depth, title):
    df_depth.groupby('name').depth_inches.plot()
    plt.legend()
    plt.title(title)
    plt.ylabel("Depth (inches)")

sensor_name_lookup = df_deployments.set_index('deployment_id').name

In [7]:
from datetime import datetime
df_depth = query_depth_data_for_multiple_deployments(
    df_deployments,
    start_time="2010-09-01T00:00:00", 
    end_time=datetime.now().isoformat()
)

# show a summary
print(len(df_depth), "results")
df_depth.head()

2907347 results


,time,deployment_id,depth_proc_mm,name_x,depth_inches,name_y,date_deployed,date_down,deploy_type,location,...,coordinates,deploy_date,deploy_time,deploy_tz,borough,intersection,street1,street2,latitude,longitude
0,2024-10-10 21:20:55.673000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.6439,-73.74448
1,2024-10-10 21:21:59.522000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.6439,-73.74448
2,2024-10-10 21:23:03.370000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.6439,-73.74448
3,2024-10-10 21:24:07.219000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.6439,-73.74448
4,2024-10-10 21:25:11.068000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.6439,-73.74448


In [8]:
df_depth

,time,deployment_id,depth_proc_mm,name_x,depth_inches,name_y,date_deployed,date_down,deploy_type,location,...,coordinates,deploy_date,deploy_time,deploy_tz,borough,intersection,street1,street2,latitude,longitude
0,2024-10-10 21:20:55.673000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.643900,-73.744480
1,2024-10-10 21:21:59.522000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.643900,-73.744480
2,2024-10-10 21:23:03.370000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.643900,-73.744480
3,2024-10-10 21:24:07.219000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.643900,-73.744480
4,2024-10-10 21:25:11.068000+00:00,definitely-neat-hippo,0.0,Q - Brookville Blvd/ Snake Rd 1,0.0,Q - Brookville Blvd/ Snake Rd 1,2024-10-10 10:00:00-04:00,NaT,coastal,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.6439, -73.74448]",2024-10-10,10:00:00,America/New_York,Q,Brookville Blvd/ Snake Rd 1,Brookville Blvd,Snake Rd 1,40.643900,-73.744480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2907342,2024-07-10 18:18:05.468000+00:00,widely_sweet_sheep,0.0,BK - 85th St 24th Ave,0.0,BK - 85th St 24th Ave,2024-06-20 00:00:00-04:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.59974610125717, -73.98847871825123]",2024-06-20,00:00:00,America/New_York,BK,85th St 24th Ave,85th St 24th Ave,NaN,40.599746,-73.988479
2907343,2024-07-10 18:19:09.214000+00:00,widely_sweet_sheep,0.0,BK - 85th St 24th Ave,0.0,BK - 85th St 24th Ave,2024-06-20 00:00:00-04:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.59974610125717, -73.98847871825123]",2024-06-20,00:00:00,America/New_York,BK,85th St 24th Ave,85th St 24th Ave,NaN,40.599746,-73.988479
2907344,2024-07-10 18:20:12.960000+00:00,widely_sweet_sheep,0.0,BK - 85th St 24th Ave,0.0,BK - 85th St 24th Ave,2024-06-20 00:00:00-04:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.59974610125717, -73.98847871825123]",2024-06-20,00:00:00,America/New_York,BK,85th St 24th Ave,85th St 24th Ave,NaN,40.599746,-73.988479
2907345,2024-07-10 18:21:16.705000+00:00,widely_sweet_sheep,0.0,BK - 85th St 24th Ave,0.0,BK - 85th St 24th Ave,2024-06-20 00:00:00-04:00,NaT,pluvial,"{'type': 'Point', 'crs': {'type': 'name', 'pro...",...,"[40.59974610125717, -73.98847871825123]",2024-06-20,00:00:00,America/New_York,BK,85th St 24th Ave,85th St 24th Ave,NaN,40.599746,-73.988479


In [9]:
df_depth['date'] = df_depth['time'].dt.date
df_depth['time_of_day'] = df_depth['time'].dt.time

df_depth = df_depth.rename(columns={'name_x': 'name'}).drop(columns=['name_y'])

In [10]:
df_got_rain = df_depth.query('depth_inches > 0.0')

In [11]:
df_got_rain.to_csv('got_rain.csv')

In [12]:
df_depth.to_csv('full_dataset.csv')